# 🚀 Google Vertex AI Native SDK 기반 Gemma 3 12B 파인튜닝 및 배포 실전 가이드

본 jupyter notebook은 **Gemma 3 12B-it** 모델을 대상으로 구글 클라우드의 완전 관리형 SFT(Supervised Fine-Tuning) API를 사용하여 모델을 미세조정하고, 배포 및 검증하는 전체 표준 워크플로우를 다룹니다.

### 💡 핵심 기술 특징 (Vertex AI Native SFT)
- **완전 관리형 SFT API (`vertexai.tuning.sft`)**: 로컬에서 Docker를 빌드하거나 복잡한 양자화 코드를 직접 구현할 필요 없이, 구글이 보장하는 최적화된 고성능 GPU 학습 인프라에서 백그라운드로 파인튜닝을 대행합니다.
- **자동 모델 등록**: SFT 완료 모델이 별도 작업 없이 **Vertex AI Model Registry**에 자동으로 등재되어 즉시 배포할 수 있는 통합 파이프라인을 지원합니다.

---

## 📦 1. 공통 환경 준비 (GCP 초기화 및 SDK 설치)

In [ ]:
# 필수 패키지 설치
from gemma4 import BUCKET_NAME
!pip install -q --upgrade google-cloud-aiplatform datasets openai

import vertexai
from google.cloud import aiplatform
import time

# GCP 프로젝트 환경 설정
PROJECT_ID = "<your project id>"
LOCATION = "asia-southeast1" # 싱가포르 리전
MACHINE_TYPE = "g4-standard-48"
ACCELERATOR_TYPE = "NVIDIA_RTX_PRO_6000"
ACCELERATOR_COUNT = 1
BUCKET_NAME="<your bucket name>"

# SDK 초기화
vertexai.init(project=PROJECT_ID, location=LOCATION)
aiplatform.init(project=PROJECT_ID, location=LOCATION)

print(f"✅ GCP SDK 연동 및 초기화 완료! (Project: {PROJECT_ID}, Location: {LOCATION})")

## 🎯 2. 순정 Gemma 3 12B IT 모델 배포 (Model Garden API)

구글 Model Garden에서 공식 배포 스펙을 갖춘 `gemma3@gemma-3-12b-it` 모델을 L4 GPU에 배포합니다.

In [ ]:
from vertexai import model_garden

MODEL_GARDEN_GEMMA3_ID = "gemma-3-12b-it"
BASE_ENDPOINT_DISPLAY_NAME = f"gemma3-base-endpoint-{int(time.time())}"

try:
    print(f"🔍 Model Garden에서 {MODEL_GARDEN_GEMMA3_ID} 스펙을 로드합니다...")
    # publishers/google/models/gemma3/gemma-3-12b-it 형식을 사용하여 로드
    model = model_garden.OpenModel(f"publishers/google/models/gemma3@{MODEL_GARDEN_GEMMA3_ID.lower()}")

    print("🚀 순정 Gemma 3 배포 프로세스를 시작합니다 (완료까지 약 15~20분 소요)...\n")
    base_endpoint = model.deploy(
        machine_type=MACHINE_TYPE,                  # NVIDIA L4 GPU 1장
        accelerator_type=ACCELERATOR_TYPE,
        accelerator_count=ACCELERATOR_COUNT,
        endpoint_display_name=BASE_ENDPOINT_DISPLAY_NAME,
        accept_eula=True                                # 라이선스 사용약관 동의
    )
    print(f"\n✅ Gemma 3 Base Model 배포 성공! Endpoint ID: {base_endpoint.resource_name}")
except Exception as e:
    print(f"❌ 배포 중 오류 발생: {e}")

## 🔍 3. Base Model 성능 확인 (추론 테스트)

배포 완료된 순정 Gemma 3 Endpoint에 BigQuery 파티션 만료에 대한 질문을 전송합니다.

In [ ]:
# 💡 [비교 대조군] 파인튜닝 전 순정 모델에 BigQuery 파티션 만료 정책을 질문합니다.
test_prompt = "사내 'Aura-FinOps' 규정에 따른 BigQuery 부서별 비용(Cost Allocation) 분류 시 필수 지정 라벨과 누락 시 제재 사항이 뭐야?"

instances = [
    {
        "@requestFormat": "chatCompletions",
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": test_prompt
                    }
                ]
            }
        ],
        "max_tokens": 2048
    }
]

print("💬 Gemma 3 Base Model Endpoint로 질문을 전송합니다...")
response = base_endpoint.predict(instances=instances)
print("\n================ [ Gemma 3 순정 모델 답변 결과 ] ================")
print(response.predictions["choices"][0]["message"]["content"])
print("========================================================")

## 🛠️ 4. Google SFT API 기반 Gemma 3 파인튜닝 (Open Model Tuning)

구글 공식 가이드에 준하여 `vertexai.tuning.sft` 패키지를 활용한 완전 관리형 QLoRA/SFT 튜닝 파이프라인을 기동합니다.

> [!IMPORTANT]
> **Vertex AI SFT 데이터셋 포맷 정합성**:
> 원본 데이터셋(`bigquery_faq_finetune.jsonl`)은 Hugging Face SFTTrainer 규격인 `{"input": "...", "output": "..."}` 키값을 따르고 있어, 이를 그대로 사용하면 SFT API에서 파싱 에러가 발생합니다.
> 따라서 구글 SFT 표준 규격인 **`{"prompt": "...", "response": "..."}`** 형태로 키값을 변환한 **`bigquery_faq_sft_ready.jsonl`** 파일을 GCS 버킷에 새로 업로드하여 학습에 활용합니다.

In [ ]:
from vertexai.tuning import sft

SOURCE_MODEL = "publishers/google/models/gemma3@gemma-3-12b-it"
TRAIN_DATASET = "gs://{BUCKET_NAME}/bigquery_faq_sft_ready.jsonl"  # input/output -> prompt/response 변환 완료 스펙
TUNED_MODEL_DISPLAY_NAME = f"gemma3-12b-it-tuned-{int(time.time())}"

print(f"🚀 Vertex AI SFT 파이닝 작업을 제출합니다...")
print(f"   - Source Model: {SOURCE_MODEL}")
print(f"   - Train Dataset: {TRAIN_DATASET}")

OUTPUT_URI = "gs://{BUCKET_NAME}/sft_output"  # HNS가 비활성화된 표준 버킷 경로  # 파인튜닝 가중치 출력 보관소 GCS 경로

sft_job = sft.train(
    source_model=SOURCE_MODEL,
    train_dataset=TRAIN_DATASET,
    output_uri=OUTPUT_URI,
    epochs=3,
    tuned_model_display_name=TUNED_MODEL_DISPLAY_NAME,
)

print(f"\n✅ SFT Tuning Job 제출 완료! Job Resource Name: {sft_job.resource_name}")
print("⏳ 백그라운드 학습이 완료될 때까지 대기합니다 (수십 분 소요)...\n")
import time
while not sft_job.has_ended:
    time.sleep(30)
    sft_job.refresh()
    print(f"   - 현재 SFT 진행 상태: {sft_job.state}")

if not sft_job.has_succeeded:
    raise RuntimeWarning(f"❌ SFT 학습이 정상적으로 종료되지 않았습니다. 상태: {sft_job.state}, 에러: {sft_job.error}")

print("\n🎉 파인튜닝 학습 완료!")
print(f"   - Tuned Model Name: {sft_job.tuned_model_name}")

## 🚀 5. 파인튜닝 완료 모델 배포 (Tuned Model Deployment)

학습 결과물로 자동 등록된 Tuned Model 리소스를 Vertex AI Endpoint에 즉시 배포합니다.

In [ ]:
print(f"📦 Model Registry로부터 튜닝된 모델({sft_job.tuned_model_name}) 정보를 읽어옵니다...")
tuned_model = aiplatform.Model(sft_job.tuned_model_name)

print("\n🚀 튜닝 완료된 모델 배포 프로세스를 기동합니다...")
tuned_endpoint = tuned_model.deploy(
    machine_type=MACHINE_TYPE,                  # NVIDIA L4 GPU 1장
    accelerator_type=ACCELERATOR_TYPE,
    accelerator_count=ACCELERATOR_COUNT,
    traffic_percentage=100,
    min_replica_count=1,
    max_replica_count=1
)
print(f"\n🎉 파인튜닝 모델 배포 완료! Active Endpoint ID: {tuned_endpoint.resource_name}")

## 💬 6. 파인튜닝 모델 최종 검증 (Tuned Model Predict)

배포된 튜닝 완료 엔드포인트를 호출하여 사내 FAQ 데이터 지식이 정상적으로 반영되었는지 검증합니다.

In [ ]:
# 💡 파인튜닝 후: 파인튜닝 데이터셋에 포함된 사내 BigQuery 전문 기술 지식을 질문하여 학습 여부를 검증합니다.
test_prompt = "사내 'Aura-FinOps' 규정에 따른 BigQuery 부서별 비용(Cost Allocation) 분류 시 필수 지정 라벨과 누락 시 제재 사항이 뭐야?"

instances = [
    {
        "@requestFormat": "chatCompletions",
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": test_prompt
                    }
                ]
            }
        ],
        "max_tokens": 2048
    }
]

print("💬 파인튜닝된 Gemma 3 Endpoint로 예측 요청을 전송합니다...")
response = tuned_endpoint.predict(instances=instances)
print("\n================ [ 🎯 Gemma 3 파인튜닝 모델 답변 결과 ] ================")
print(response.predictions[0][0]["message"]["content"])
print("========================================================================")

## 🧹 7. 비용 방지를 위한 자원 일괄 삭제 (Clean-up)

데모가 완료되면 불필요하게 가동 중인 GPU Endpoint를 신속하게 중단시킵니다.

In [ ]:
print("🧹 Gemma 3 관련 활성 엔드포인트 삭제 시작...")
endpoints = aiplatform.Endpoint.list(order_by="create_time desc")
for ep in endpoints:
    if "gemma3" in ep.display_name:
        print(f"   - Undeploying and deleting endpoint: {ep.display_name}")
        ep.delete(force=True)

print("\n🧹 Gemma 3 관련 등록 모델 삭제 시작...")
models = aiplatform.Model.list(order_by="create_time desc")
for m in models:
    if "gemma3" in m.display_name:
        print(f"   - Deleting model: {m.display_name}")
        m.delete()

print("\n🎉 Gemma 3 관련 가동 인프라 자원이 안전하게 완전 정리되었습니다!")